In [1]:
import time
import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder

class RerankerEvaluator:
    def __init__(self):
        print("Loading BGE Reranker (BAAI/bge-reranker-base)...")
        self.bge_model = CrossEncoder('BAAI/bge-reranker-base')
        
        print("Loading Baseline Cross-Encoder (cross-encoder/ms-marco-MiniLM-L-6-v2)...")
        self.marco_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    def calculate_mrr(self, predicted_ranks, expected_relevant_id):
        """Calculates Reciprocal Rank for a single query."""
        for rank, item in enumerate(predicted_ranks):
            if item["chunk_id"] == expected_relevant_id:
                return 1.0 / (rank + 1)
        return 0.0

    def calculate_precision_at_k(self, predicted_ranks, expected_relevant_id, k=3):
        """Calculates Precision@K (Binary relevance for this specific test)."""
        top_k_ids = [item["chunk_id"] for item in predicted_ranks[:k]]
        return 1.0 if expected_relevant_id in top_k_ids else 0.0

    def evaluate(self, eval_dataset: list):
        results = []
        
        for model_name in ["BGE-Reranker", "MS-MARCO"]:
            print(f"Evaluating {model_name} across {len(eval_dataset)} samples...")
            model_mrr = []
            model_p_at_3 = []
            total_time = 0
            
            for data in eval_dataset:
                query = data["query"]
                candidates = data["candidates"]
                expected_id = data["relevant_chunk_id"]
                
                sentence_pairs = [[query, item["text"]] for item in candidates]
                
                start_time = time.time()
                if model_name == "BGE-Reranker":
                    scores = self.bge_model.predict(sentence_pairs)
                else:
                    scores = self.marco_model.predict(sentence_pairs)
                total_time += (time.time() - start_time)
                
                if isinstance(scores, float) or isinstance(scores, np.float32):
                    scores = [scores]
                    
                ranked_candidates = [
                    {"chunk_id": candidates[i]["chunk_id"], "score": scores[i]} 
                    for i in range(len(candidates))
                ]
                ranked_candidates.sort(key=lambda x: x["score"], reverse=True)
                
                # Record Metrics
                model_mrr.append(self.calculate_mrr(ranked_candidates, expected_id))
                model_p_at_3.append(self.calculate_precision_at_k(ranked_candidates, expected_id, k=3))
            
            # Aggregate Results
            results.append({
                "Model": model_name,
                "Mean MRR": round(np.mean(model_mrr), 3),
                "Mean Precision@3": round(np.mean(model_p_at_3), 3),
                "Avg Latency/Query (ms)": round((total_time / len(eval_dataset)) * 1000, 2)
            })
            
        return pd.DataFrame(results)
# --- ADVERSARIAL BENCHMARK DATASET ---
evaluation_dataset = [
    {
        "query": "Is trastuzumab deruxtecan indicated as second-line therapy for HER2-low metastatic breast cancer?",
        "relevant_chunk_id": "correct_second_line",
        "candidates": [
            {"chunk_id": "wrong_first_line", "text": "Trastuzumab deruxtecan is now recommended as preferred first-line systemic therapy for HER2-low metastatic breast cancer."},
            {"chunk_id": "wrong_third_line", "text": "For patients progressing after multiple targeted regimens, trastuzumab deruxtecan may be considered as a third-line or later option."},
            {"chunk_id": "correct_second_line", "text": "Trastuzumab deruxtecan is designated for patients with HER2-low unresectable or metastatic breast cancer who have received a prior chemotherapy regimen in the metastatic setting."}
        ]
    },
    {
        "query": "Preferred choice for advanced EGFR exon 20 insertion mutations in NSCLC",
        "relevant_chunk_id": "correct_exon_20",
        "candidates": [
            {"chunk_id": "wrong_exon_19", "text": "Osimertinib is a highly effective, preferred first-line tyrosine kinase inhibitor targeting classic EGFR exon 19 deletions."},
            {"chunk_id": "correct_exon_20", "text": "Amivantamab-vmjw or mobocertinib are specific therapeutic choices recommended for non-small cell lung cancer characterized by EGFR exon 20 insertion mutations."},
            {"chunk_id": "wrong_l858r", "text": "For advanced non-small cell lung cancer harboring EGFR L858R point mutations, preferred initial therapy options include standard third-generation TKIs."}
        ]
    },
    {
        "query": "Adjuvant systemic therapy options for resectable stage III melanoma with BRAF V600E mutation",
        "relevant_chunk_id": "correct_adjuvant",
        "candidates": [
            {"chunk_id": "wrong_metastatic", "text": "In the unresectable or metastatic setting, combination therapy with dabrafenib and trametinib is a standard option for BRAF V600E mutated melanoma."},
            {"chunk_id": "correct_adjuvant", "text": "Preferred options for adjuvant treatment of resected stage III melanoma containing a BRAF V600E/K mutation include dabrafenib plus trametinib or pembrolizumab."},
            {"chunk_id": "wrong_neo_adjuvant", "text": "Neoadjuvant treatment options for high-risk resectable melanoma are being evaluated to downstage tumors prior to surgical resection."}
        ]
    }
]
# --- EXPANDED BENCHMARK DATASET ---
'''
evaluation_dataset = [
    {
        "query": "What is the absolute risk of pancreatic cancer for an ATM mutation?",
        "relevant_chunk_id": "atm_panc",
        "candidates": [
            {"chunk_id": "atm_rad", "text": "Heterozygous ATM variants should not avoid radiation therapy."},
            {"chunk_id": "ovarian_risk", "text": "Epithelial Ovarian Cancer Risk. Absolute risk: 2%-3%."},
            {"chunk_id": "atm_breast", "text": "ATM Absolute risk: 20%-40% Management: Annual breast MRI."},
            {"chunk_id": "atm_panc", "text": "Pancreatic cancer • Absolute risk: ~5%-10% • Management: Screen variant carriers with family history."}
        ]
    },
    {
        "query": "Targeted therapies recommended for KRAS G12C mutations in NSCLC",
        "relevant_chunk_id": "g12c_correct",
        "candidates": [
            {"chunk_id": "g12d_wrong", "text": "Adagrasib trial updates demonstrate minimal efficacy in KRAS G12D variants."},
            {"chunk_id": "g12c_correct", "text": "Sotorasib and Adagrasib are FDA-approved targeted inhibitors specifically for advanced KRAS G12C mutated NSCLC."},
            {"chunk_id": "egfr_wrong", "text": "Osimertinib remains the standard of care for EGFR exon 19 deletions in non-small cell lung cancer."}
        ]
    },
    {
        "query": "Is trastuzumab deruxtecan indicated for HER2-low metastatic breast cancer?",
        "relevant_chunk_id": "her2_low_true",
        "candidates": [
            {"chunk_id": "her2_pos", "text": "Trastuzumab, pertuzumab, and docetaxel are standard first-line therapies for classic HER2-positive (IHC 3+) metastatic disease."},
            {"chunk_id": "her2_low_true", "text": "Fam-trastuzumab deruxtecan-nxki is an option for patients with HER2-low (IHC 1+ or IHC 2+/ISH-) unresectable or metastatic breast cancer who received prior chemotherapy."},
            {"chunk_id": "tnbc_wrong", "text": "Sacituzumab govitecan is an antibody-drug conjugate approved for metastatic triple-negative breast cancer."}
        ]
    },
    {
        "query": "First-line preferred systemic therapy options for advanced clear cell renal cell carcinoma",
        "relevant_chunk_id": "rcc_first_line",
        "candidates": [
            {"chunk_id": "rcc_second_line", "text": "Cabozantinib or lenvatinib + everolimus are common subsequent or second-line regimens for clear cell renal cell carcinoma."},
            {"chunk_id": "rcc_first_line", "text": "Preferred first-line category 1 regimens for advanced clear cell RCC include Axitinib + Pembrolizumab and Cabozantinib + Nivolumab."},
            {"chunk_id": "non_clear_cell", "text": "Sunitinib is a preferred option for non-clear cell renal cell carcinoma."}
        ]
    }
]
'''
# Run evaluation
evaluator = RerankerEvaluator()
df_report = evaluator.evaluate(evaluation_dataset)

print("\n--- PHASE 4: REFINED RERANKING PERFORMANCE REPORT ---")
display(df_report)

Loading BGE Reranker (BAAI/bge-reranker-base)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading Baseline Cross-Encoder (cross-encoder/ms-marco-MiniLM-L-6-v2)...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Evaluating BGE-Reranker across 3 samples...
Evaluating MS-MARCO across 3 samples...

--- PHASE 4: REFINED RERANKING PERFORMANCE REPORT ---


,Model,Mean MRR,Mean Precision@3,Avg Latency/Query (ms)
0,BGE-Reranker,1.000,1.0,61.77
1,MS-MARCO,0.833,1.0,11.74
